# YOLOv8 Baseline - Colab

Train, evaluate, and run inference with YOLOv8n on Google Drive data.
Setup cells mirror `colab_template.ipynb`.

In [1]:
REPO = "road-damage-detection"

# Clone the repository (skip if already cloned)
!test -d $REPO || git clone https://github.com/orzmik/road-damage-detection.git
%cd $REPO

# Install dependencies needed for Colab runs
!pip -q install ultralytics wandb

Cloning into 'road-damage-detection'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 138 (delta 59), reused 116 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 2.40 MiB | 31.46 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/road-damage-detection
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.5 MB/s eta 0:00:00


In [2]:
BRANCH = "feature/refactor-training-colab"

!git fetch origin
!git checkout $BRANCH
!git pull origin $BRANCH

Branch 'feature/refactor-training-colab' set up to track remote branch 'feature/refactor-training-colab' from 'origin'.
Switched to a new branch 'feature/refactor-training-colab'
From https://github.com/orzmik/road-damage-detection
 * branch            feature/refactor-training-colab -> FETCH_HEAD
Already up to date.


In [3]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

In [12]:
# My Drive:
DRIVE_ROOT = "MyDrive/road_damage_detection"

# Shared drive (teammates):
# DRIVE_ROOT = "Shareddrives/<TeamDrive>/road-damage-detection"

In [13]:
from src.config.colab.drive import DriveConfig, ensure_drive_paths, mount_drive

mount_drive()

drive_cfg = DriveConfig(drive_root=DRIVE_ROOT)
paths = ensure_drive_paths(drive_cfg)

print(f"Drive root:       {paths.root}")
print(f"Processed data:   {paths.processed_yolo}")
print(f"Models:           {paths.models}")
print(f"Training runs:    {paths.runs}")
print(f"Wroclaw images:   {paths.wroclaw_images}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root:       /content/drive/MyDrive/road_damage_detection
Processed data:   /content/drive/MyDrive/road_damage_detection/data/processed-yolo
Models:           /content/drive/MyDrive/road_damage_detection/models
Training runs:    /content/drive/MyDrive/road_damage_detection/runs
Wroclaw images:   /content/drive/MyDrive/road_damage_detection/wroclaw_images


## Training

In [8]:
import wandb
from google.colab import userdata


wandb.login(key=userdata.get('WANDB_API_KEY'))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: koostosh (ADM-lists) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [15]:
from src.config.yolo import create_yolo_data_yaml, train_yolo

import shutil
from pathlib import Path

LOCAL_DATA = Path("/content/data/processed-yolo")
DRIVE_DATA = paths.processed_yolo  # your Drive path
if not LOCAL_DATA.exists():
    print("Copying dataset from Drive to local disk (one-time per session)...")
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)
    print("Done.")
else:
    print("Local copy already exists, skipping copy.")
# Point YOLO at local data, not Drive
data_yaml = create_yolo_data_yaml(
    Path("/content/data/road_damage_local.yaml"),
    LOCAL_DATA,
)

Copying dataset from Drive to local disk (one-time per session)...
Done.


In [16]:
from src.config.wandb import WandbConfig

wandb_cfg = WandbConfig(
    project="road-damage-classification",
    entity="project-nn",
    name="yolov8n_colab",
    job_type="train",
    config={
        "epochs": 50,
        "imgsz": 640,
        "batch": 16,
    },
)

train_out = train_yolo(
    weights="yolov8n.pt",
    data_yaml=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=8,
    device=0,
    project_dir=paths.runs,
    run_name="yolov8n_colab",
    wandb_cfg=wandb_cfg,
)

train_out.save_dir

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/road_damage_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_colab-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,

lr/pg0,▆████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
lr/pg1,▃▆███▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
lr/pg2,▃▆███▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁
metrics/mAP50(B),▁▁▁▂▃▅▅▅▅▆▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█▇████████
metrics/mAP50-95(B),▁▁▂▁▂▄▅▅▅▅▅▅▅▆▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇███████
metrics/precision(B),█▁▂▂▂▃▄▃▄▃▄▄▃▄▄▄▄▅▅▄▅▅▅▅▅▅▅▅▆▅▅▅▆▆▅▆▆▆▆▆
metrics/recall(B),▁▃▄▄▄▆▆▆▆▆▆▇▆▇▆▇▇▆▇▆▇▇▇▇▇█▇██▇█████▇█▇██
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


PosixPath('/content/drive/MyDrive/road_damage_detection/runs/yolov8n_colab-2')

## Evaluation (test set)

In [17]:
from src.config.yolo import evaluate_yolo

best_weights = train_out.save_dir / "weights" / "best.pt"
metrics = evaluate_yolo(
    weights=best_weights,
    data_yaml=data_yaml,
    split="test",
    device=0,
    project_dir=paths.runs,
    run_name="yolov8n_colab_test",
)
metrics

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2273.2±907.0 MB/s, size: 99.2 KB)
val: Scanning /content/data/processed-yolo/test/labels... 202 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 202/202 2.3Kit/s 0.1s
val: New cache created: /content/data/processed-yolo/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 5.1it/s 2.6s
                   all        202        493      0.581      0.459      0.481      0.204
               Pothole         77        129      0.537      0.386       0.41      0.168
                 Crack        140        272      0.431      0.312      0.324      0.121
               Manhole         70         92      0.776      0.677       0.71      0.322
Speed: 1.7ms preprocess, 3.2ms inference, 0.0ms loss, 2.1ms pos

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d10c8842b70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

## Inference (Wroclaw images)

In [18]:
from src.config.yolo import predict_yolo

predict_results = predict_yolo(
    weights=best_weights,
    source=paths.wroclaw_images,
    device=0,
    project_dir=paths.runs,
    run_name="yolov8n_colab_infer",
    save=True,
)

predict_results[:2]


image 1/5 /content/drive/MyDrive/road_damage_detection/wroclaw_images/1779554991904.jpg: 288x640 1 Pothole, 1 Crack, 39.1ms
image 2/5 /content/drive/MyDrive/road_damage_detection/wroclaw_images/1779554991907.jpg: 288x640 1 Crack, 8.3ms
image 3/5 /content/drive/MyDrive/road_damage_detection/wroclaw_images/1779554991911.jpg: 288x640 1 Pothole, 12.1ms
image 4/5 /content/drive/MyDrive/road_damage_detection/wroclaw_images/1779554991914.jpg: 288x640 3 Potholes, 6.6ms
image 5/5 /content/drive/MyDrive/road_damage_detection/wroclaw_images/1779554991917.jpg: 288x640 1 Pothole, 6.4ms
Speed: 2.3ms preprocess, 14.5ms inference, 1.5ms postprocess per image at shape (1, 3, 288, 640)
Results saved to /content/drive/MyDrive/road_damage_detection/runs/yolov8n_colab_infer


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Pothole', 1: 'Crack', 2: 'Manhole'}
 obb: None
 orig_img: array([[[ 45,  89,  58],
         [ 41,  85,  54],
         [ 40,  84,  53],
         ...,
         [179, 178, 182],
         [179, 178, 182],
         [180, 179, 183]],
 
        [[ 44,  88,  57],
         [ 37,  81,  50],
         [ 36,  80,  49],
         ...,
         [182, 181, 185],
         [183, 182, 186],
         [183, 182, 186]],
 
        [[ 46,  90,  59],
         [ 40,  84,  53],
         [ 39,  83,  52],
         ...,
         [181, 180, 184],
         [179, 178, 182],
         [178, 177, 181]],
 
        ...,
 
        [[125, 127, 128],
         [125, 127, 128],
         [125, 127, 128],
         ...,
         [148, 150, 150],
         [145, 147, 147],
         [146, 148, 148]],
 
        [[127, 129, 130],
         [129, 131, 132],
         [131, 133, 134],
     